# Lab 17-02 · Noise Budgets and Calibrated AWGN

Five parts, each built on `noisebudget.py`. **Predict first**, then run. The handout is [README.md](README.md); Part 6 (ngspice) runs outside the notebook.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.stats import norm
import noisebudget as nb
rng = np.random.default_rng(1702)
print(f"kT0 = {nb.ktb_dbm(1):.2f} dBm/Hz")
print(f"NR UE REFSENS (n78, 52 RB at 15 kHz): {nb.sensitivity_dbm(52 * 12 * 15e3, 10, -1, 2.5, 3):.2f} dBm")

kT0 = -173.98 dBm/Hz
NR UE REFSENS (n78, 52 RB at 15 kHz): -95.79 dBm


## Part 1 · Noise in a bandwidth

*Predict:* the power of white noise of one-sided density $kT_0$ after ideal filters of 1, 10 and 100 kHz.

In [2]:
fs, n = 1e6, 1 << 20
N0 = nb.K_B * nb.T0
x = math.sqrt(N0 * fs / 2) * rng.standard_normal(n)     # one-sided density N0 over 0..fs/2
X = np.fft.rfft(x)
f = np.fft.rfftfreq(n, 1 / fs)
for B in (1e3, 1e4, 1e5):
    y = np.fft.irfft(np.where(f < B, X, 0), n)
    meas, pred = float(nb.dbm(np.mean(y ** 2))), nb.ktb_dbm(B)
    print(f"B = {B / 1e3:5.0f} kHz: predicted {pred:.2f} dBm, measured {meas:.2f} dBm")
    assert abs(meas - pred) < 0.15

B =     1 kHz: predicted -143.98 dBm, measured -143.93 dBm
B =    10 kHz: predicted -133.98 dBm, measured -133.99 dBm
B =   100 kHz: predicted -123.98 dBm, measured -123.98 dBm


## Part 2 · A cascade, simulated (Example 17-02.3)

*Predict:* the noise figure with the 1.5 dB cable before the LNA, and with the LNA at the antenna.

In [3]:
Fc, Tc, Gc = nb.lossy(1.5)
T_lna, G_lna = nb.te_from_nf(0.6), nb.undb(20.2)
T_rx = nb.te_from_nf(8.0)
orders = {"cable first": [(Tc, Gc), (T_lna, G_lna), (T_rx, 1.0)],
          "LNA first": [(T_lna, G_lna), (Tc, Gc), (T_rx, 1.0)]}
for name, st in orders.items():
    F_sim = nb.cascade_sim(st, 1 << 21, rng)
    F_th = 1 + nb.cascade_te([t for t, _ in st], [g for _, g in st]) / nb.T0
    print(f"{name:12s}: Friis {10 * math.log10(F_th):.3f} dB, simulated {10 * math.log10(F_sim):.3f} dB")
    assert abs(10 * math.log10(F_sim / F_th)) < 0.03

cable first : Friis 2.288 dB, simulated 2.295 dB
LNA first   : Friis 0.877 dB, simulated 0.879 dB


## Part 3 · $E_b/N_0$ does not care about the sample rate

*Predict:* the BPSK bit-error rate at $E_b/N_0$ = 6 dB, and what happens if the per-sample SNR is set to 6 dB instead.

In [4]:
theory = norm.sf(math.sqrt(2 * nb.undb(6)))
print(f"theory Q(sqrt(2 Eb/N0)) = {theory:.3e}")
for sps in (1, 4, 16):
    ber = nb.bpsk_ber_sim(6.0, sps, 2_000_000, rng)
    print(f"{sps:2d} samples per bit: BER {ber:.3e}")
    assert abs(ber / theory - 1) < 0.06
wrong = nb.bpsk_ber_sim(6.0, 8, 2_000_000, rng, per_sample_snr_mistake=True)
print(f"per-sample SNR set to 6 dB at 8 samples per bit: BER {wrong:.3e} (that run is really at {6 + 10 * math.log10(8 / 2):.2f} dB, "
      f"theory {norm.sf(math.sqrt(2 * nb.undb(6 + 10 * math.log10(4)))):.1e})")
assert wrong < 1e-5

theory Q(sqrt(2 Eb/N0)) = 2.388e-03


 1 samples per bit: BER 2.436e-03


 4 samples per bit: BER 2.393e-03


16 samples per bit: BER 2.446e-03


per-sample SNR set to 6 dB at 8 samples per bit: BER 0.000e+00 (that run is really at 12.02 dB, theory 8.3e-09)


## Part 4 · Below the noise

*Predict:* the SNR after correlating 1023 chips received at −20 dB per chip.

In [5]:
snr_in, snr_out = nb.despread_snr(-20.0, 1023, 4000, rng)
print(f"per-chip SNR {snr_in:.2f} dB -> after correlation {snr_out:.2f} dB (predicted {-20 + 10 * math.log10(1023):.2f} dB)")
assert abs(snr_out - (-20 + 10 * math.log10(1023))) < 0.3

per-chip SNR -20.00 dB -> after correlation 9.96 dB (predicted 10.10 dB)


## Part 5 · Noise bandwidth of a one-pole filter

*Predict:* how much more noise passes a one-pole filter than its 3 dB bandwidth suggests.

In [6]:
fs, fc, n = 1e6, 1e3, 1 << 23
N0 = 1.0                                   # one-sided density, arbitrary units
x = math.sqrt(N0 * fs / 2) * rng.standard_normal(n)
a = math.exp(-2 * math.pi * fc / fs)
y = signal.lfilter([1 - a], [1, -a], x)[20000:]
ratio = np.mean(y ** 2) / (N0 * fc)
print(f"measured noise / (N0 fc) = {ratio:.3f} ({10 * math.log10(ratio):.2f} dB); pi/2 = {math.pi / 2:.3f} (1.96 dB)")
assert abs(10 * math.log10(ratio) - 1.96) < 0.1

measured noise / (N0 fc) = 1.570 (1.96 dB); pi/2 = 1.571 (1.96 dB)


## Part 6 · ngspice

Run `python tools/run_spice.py 17-02` from the repository root. The 50 Ω source delivers −113.98 dBm in 1 MHz to the matched load, and the 6 dB pad has a 6.00 dB loss and a 6.00 dB noise figure.